In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:
%run /Workspace/Users/vaishnavikumbhakarna467@gmail.com/Fast-Moving-Consumer-Good-Databricks/01_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","catalog")
dbutils.widgets.text("data_source","customers","Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")


base_path = f's3://sportsbar-vk-01/{data_source}/*.csv'
print(base_path)

In [0]:
df = (
    spark.read.format('csv')
    .option('header',True)
    .option('inferschema',True)
    .load(base_path)
    .withColumn('read_timestamp',F.current_timestamp())
    .select("*","_metadata.file_name","_metadata.file_size")
)
display(df.limit(10))

In [0]:
df.write\
    .format('delta') \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
df.printSchema()

**Silver Processing**

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(5)

In [0]:
df_bronze.printSchema()

In [0]:
print("Rows before dropping duplicates: ",df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Rows After dropping duplicates: ",df_silver.count())

In [0]:
# to check the customers having leading spaces in their names

display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col('customer_name')))
)

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name")))

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col('customer_name')))
)

In [0]:
#  cities name should be distinct and of same spelling

In [0]:
# capital small case fix